# Session 2 — A/B Testing & Canary Deployments

**Goal:** Understand how to safely roll out new model versions using traffic splitting.

**Canary deployment**: Send a small % of traffic to the new model before full rollout.
- Risk: If the canary has issues, only 10% of users are affected
- Monitoring window: Watch metrics for 24h before promoting to 100%

**A/B test**: Compare two model versions on business metrics (not just offline F1).
- Split: 50/50 traffic to version A and version B
- Measure: Which version drives better customer retention outcomes?

> **Workshop note:** We'll review the SDK code and walk through the Databricks UI steps —
> no endpoint updates will be executed here to save time.

In [0]:
%run ../utils/config

In [0]:

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

endpoint_name = f"churn_{safe_username}"
model_name    = f"{catalog}.{schema}.churn_classifier"

# Check what versions we have
from mlflow import MlflowClient
mlflow_client = MlflowClient()
mlflow_client.set_registry_uri = lambda x: None

import mlflow
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
versions = client.search_model_versions(f"name='{model_name}'")
print(f"Available versions for {model_name}:")
for v in versions:
    # search_model_versions doesn't include aliases; use get_model_version instead
    mv = client.get_model_version(model_name, v.version)
    alias_list = mv.aliases if not callable(mv.aliases) else []
    aliases = ", ".join(alias_list) if alias_list else "(none)"
    print(f"  v{v.version}  aliases=[{aliases}]  status={v.status}")

## Review the Canary Configuration

The cell below builds the endpoint config for a 90/10 canary split and prints it as JSON.
This is exactly what the SDK sends to the Databricks API — you can also apply it directly
in the UI (see instructions below).

> **Pre-requisite:** At least 2 registered model versions are needed for a canary split.
> If the retrain job from the previous notebook completed, you should have v2 available.

In [ ]:
# Get the two most recent versions
all_versions = sorted(versions, key=lambda v: int(v.version))

if len(all_versions) < 2:
    print("⚠️  Only 1 model version found — the retrain job from the previous notebook")
    print("    may still be running. Showing an illustrative config with a placeholder v2.")
    v1 = all_versions[-1].version
    v2 = str(int(v1) + 1)   # placeholder — not a real version yet
else:
    v1 = all_versions[-2].version  # Previous stable version
    v2 = all_versions[-1].version  # New canary version
    print(f"Stable version  : v{v1}")
    print(f"Canary version  : v{v2}")

print(f"\nIllustrative config: 90% to v{v1}, 10% to v{v2}")

import json
endpoint_config = {
    "served_models": [
        {"model_name": model_name, "model_version": str(v1),
         "workload_size": "Small", "scale_to_zero_enabled": True, "name": f"v{v1}"},
        {"model_name": model_name, "model_version": str(v2),
         "workload_size": "Small", "scale_to_zero_enabled": True, "name": f"v{v2}"},
    ],
    "traffic_config": {
        "routes": [
            {"served_model_name": f"v{v1}", "traffic_percentage": 90},
            {"served_model_name": f"v{v2}", "traffic_percentage": 10},
        ]
    }
}
print(json.dumps(endpoint_config, indent=2))

## Performing a Canary Deployment in the Databricks UI

The SDK config above shows exactly what the API sends. You can do the same thing in the UI without writing any code:

**Step 1 — Add the canary version:**
1. Go to **Serving** in the left sidebar
2. Click your `churn_<safe_username>` endpoint
3. Click **Edit configuration**
4. Click **Add served entity** and select `churn_classifier` version 2
5. Set **Traffic %**: v1 = `90`, v2 = `10`
6. Click **Update** and wait for the endpoint to reach Ready state

**Step 2 — Promote to 50/50 (after monitoring looks healthy):**
1. Return to **Edit configuration**
2. Set **Traffic %**: v1 = `50`, v2 = `50`
3. Click **Update**

**Step 3 — Full promotion or rollback:**

| Decision | Action |
|---|---|
| Canary metrics look good | Set v2 = 100%, remove v1 |
| Canary metrics degraded | Set v1 = 100%, remove v2 |

> **Rollback is instantaneous** — just shift all traffic back to v1 and update. No redeployment needed.

## Discussion

- **How long should you monitor the canary before promoting to 100%?** (Depends on traffic volume — you need enough requests for statistical significance)
- **What business metrics would you track alongside PSI drift?** (Conversion rate, revenue per prediction, customer complaint rate)
- **Who approves the final promotion from 10% → 100%?** (Data scientist? MLOps engineer? Product owner?)

➡️ Next: `07_incident_runbook.ipynb` *(optional bonus)* — walk through a full incident response